# v19 — Strategi Terpisah untuk Rezim Volatilitas Rendah (2019-2024)

**Latar belakang:** v18 membuktikan strategi v13 (dituning & divalidasi di rezim volatilitas
tinggi 2025-2026) TIDAK COCOK dipaksakan ke rezim volatilitas rendah -- dengan spread broker
realistis (1.82), filter "ATR minimal 3x spread" secara efektif men-skip HAMPIR SEMUA entry di
2019-2024 (cuma tersisa 55 dari 189 trade total 2019-2026 yang lolos, sisanya 134 di 2026 saja).
**Keputusan (bukan bug, tapi keputusan disain):** daripada memaksakan 1 formula utk semua rezim,
robot SEHARUSNYA tidak trading sama sekali di rezim volatilitas rendah dgn parameter v13 --
tapi itu berarti kehilangan seluruh peluang di rezim itu. v19 mencari APAKAH ada pola/parameter
BERBEDA yang justru cocok utk karakteristik rezim volatilitas rendah, supaya robot punya 2
strategi: v13 utk rezim tinggi (skrg aktif), + strategi baru utk rezim rendah (kalau ditemukan
valid) -- bukan diam total kalau market kembali ke kondisi spt 2019-2024.

**Cakupan data**: seluruh 2019-2024 (rezim volatilitas rendah, SEBELUM 2025 -- 2025-2026
sengaja dikecualikan krn itu wilayah v13 sudah terbukti bekerja baik). Displit TRAIN (2019-2023,
351K candle) & TEST (2024, 60K candle, out-of-sample) -- metodologi sama persis dgn semua riset
sebelumnya (v06-v13, v18), supaya parameter yang ditemukan bukan overfitting ke seluruh rezim
sekaligus.

**Pendekatan**: REUSE mesin scoring v12 (20 kategori indikator, cluster oscillator/trend-follower
yang sudah divalidasi) -- BUKAN eksplorasi logika sinyal dari nol. Yang dicari ulang khusus utk
karakter rezim rendah:
1. **`MIN_SIGNAL_SCORE`** -- mungkin threshold 9.0 (dituning utk rezim tinggi) terlalu ketat/longgar
   utk distribusi skor di rezim rendah
2. **SL/TP multiplier** -- ATR absolut jauh lebih kecil di rezim rendah, mungkin butuh kelipatan
   ATR yang beda drpd 2x/4x supaya margin thd spread tetap sehat
3. **Filter ATR-vs-spread minimum** (dari v18) -- threshold berapa yang PALING PAS utk rezim ini
   spesifik (bukan 3.0 yang dicari dari rezim TINGGI di v18)
4. Kombinasi ketiganya lewat grid search di TRAIN, divalidasi independen di TEST

**Kriteria keberhasilan**: strategi baru dianggap LAYAK kalau (a) profit factor > 1.5 di TEST
out-of-sample DENGAN spread realistis 1.82, (b) sample trade cukup besar utk dipercaya (>=30
trade TRAIN, >=15 trade TEST -- pelajaran dari bug grid search v18 yg sempat overfitting ke 6
trade), (c) hasilnya BUKAN "skip semua" spt yang sudah dibuktikan gagal di v18. Kalau kriteria
ini tidak terpenuhi, kesimpulan JUJUR-nya adalah "rezim volatilitas rendah memang tidak layak
ditradingkan dgn EDGE apa pun yang bisa ditemukan dari indikator yang ada" -- bukan dipaksakan.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v19"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82  # spread broker MIFX riil, dari v18

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Scoring ULANG tanpa hard filter ADX/ATR (supaya bisa eksplorasi ulang threshold-nya juga)

Cache v18 (`v12_raw_score`) sudah difilter `ADX>=18` & `ATR%>=0.03` (sama spt v13) SEBELUM skor
dihitung -- utk grid search yang lebih menyeluruh (termasuk cari ulang APAKAH threshold ADX/ATR
minimum itu juga perlu beda utk rezim rendah), skor perlu dihitung utk SEMUA candle dulu, filter
diterapkan BELAKANGAN sbg parameter yang bisa divariasikan.

In [2]:
SCORED_ALL_CACHE_PATH = PROCESSED_DIR / "v19" / "df_2019_2024_scored_nofilter.parquet"
(PROCESSED_DIR / "v19").mkdir(parents=True, exist_ok=True)

if SCORED_ALL_CACHE_PATH.exists():
    print(f"Load skor dari cache: {SCORED_ALL_CACHE_PATH}")
    df_scored = pd.read_parquet(SCORED_ALL_CACHE_PATH)
else:
    print("Belum ada cache -- hitung skor SEMUA candle 2019-2024 (tanpa filter ADX/ATR)...")
    import time as _time

    from app.utils.signals.scoring import (
        score_adx, score_bb, score_candle, score_cci, score_extra_patterns,
        score_fibonacci, score_ichimoku, score_macd, score_mfi, score_momentum_chain,
        score_obv, score_psar, score_rsi, score_rsi_divergence, score_sma, score_smc,
        score_stoch, score_supertrend, score_vwap, score_williams_r,
    )

    OSCILLATOR_CLUSTER = ["rsi", "stoch", "williams_r", "cci", "bb", "vwap"]
    TREND_CLUSTER = ["sma", "ichimoku", "supertrend"]
    OLD_WEIGHTS = {
        "rsi": 1.0, "stoch": 1.0, "williams_r": 1.0, "cci": 1.0, "bb": 1.0, "vwap": 1.5,
        "sma": 1.5, "ichimoku": 2.5, "supertrend": 2.5,
    }
    OSCILLATOR_WEIGHT = round(np.mean([1.0, 1.0, 1.0, 1.0, 1.0, 1.5]), 2)
    TREND_WEIGHT = round(np.mean([1.5, 2.5, 2.5]), 2)

    def score_de_redundant(row):
        close = float(row["close"])
        atr = float(row.get("atr", close * 0.001))
        raw_scores = {
            "rsi": score_rsi(row)[0] / OLD_WEIGHTS["rsi"],
            "stoch": score_stoch(row)[0] / OLD_WEIGHTS["stoch"],
            "williams_r": score_williams_r(row)[0] / OLD_WEIGHTS["williams_r"],
            "cci": score_cci(row)[0] / OLD_WEIGHTS["cci"],
            "bb": score_bb(row, close)[0] / OLD_WEIGHTS["bb"],
            "vwap": score_vwap(row, close)[0] / OLD_WEIGHTS["vwap"],
            "sma": score_sma(row, close)[0] / OLD_WEIGHTS["sma"],
            "ichimoku": score_ichimoku(row, close)[0] / OLD_WEIGHTS["ichimoku"],
            "supertrend": score_supertrend(row)[0] / OLD_WEIGHTS["supertrend"],
        }
        oscillator_raw = np.median([raw_scores[c] for c in OSCILLATOR_CLUSTER])
        trend_raw = np.median([raw_scores[c] for c in TREND_CLUSTER])
        components = {
            "oscillator_composite": oscillator_raw * OSCILLATOR_WEIGHT,
            "trend_composite": trend_raw * TREND_WEIGHT,
            "macd": score_macd(row)[0],
            "adx": score_adx(row)[0],
            "candle": score_candle(row)[0],
            "extra_patterns": score_extra_patterns(row)[0],
            "obv": score_obv(row)[0],
            "mfi": score_mfi(row)[0],
            "fibonacci": score_fibonacci(row, close, atr)[0],
            "rsi_divergence": score_rsi_divergence(row)[0],
            "momentum_chain": score_momentum_chain(row)[0],
            "psar": score_psar(row)[0],
            "smc": score_smc(row)[0],
        }
        return round(sum(components.values()), 3)

    df_m5 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv")
    df_m5["datetime"] = pd.to_datetime(df_m5["datetime"])
    df_m5 = df_m5.sort_values("datetime").reset_index(drop=True)
    df_m5 = df_m5[df_m5["datetime"] < pd.Timestamp("2025-01-01", tz="UTC")].reset_index(drop=True)

    df_h1 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_h1_full_indicators.csv")
    df_h1["datetime"] = pd.to_datetime(df_h1["datetime"])
    df_h1 = df_h1.sort_values("datetime").reset_index(drop=True)

    df_h1_shifted = df_h1.copy()
    df_h1_shifted["h1_available_at"] = df_h1_shifted["datetime"] + pd.Timedelta(hours=1)
    h1_cols = [c for c in df_h1_shifted.columns if c != "h1_available_at"]
    df_h1_shifted = df_h1_shifted[["h1_available_at", *h1_cols]].rename(columns={c: f"h1_{c}" for c in h1_cols})

    df_scored = pd.merge_asof(
        df_m5.sort_values("datetime"), df_h1_shifted.sort_values("h1_available_at"),
        left_on="datetime", right_on="h1_available_at", direction="backward",
    )
    print(f"Dataset 2019-2024: {len(df_scored)} baris, {df_scored['datetime'].min()} -> {df_scored['datetime'].max()}")

    t0 = _time.time()
    scores = np.full(len(df_scored), np.nan)
    for pos in range(len(df_scored)):
        row = df_scored.iloc[pos]
        scores[pos] = score_de_redundant(row)
        if pos % 100_000 == 0:
            print(f"  progress: {pos}/{len(df_scored)} ({_time.time()-t0:.0f}s)")

    df_scored["v12_raw_score_nofilter"] = scores
    print(f"Scoring done in {_time.time()-t0:.0f}s (TANPA hard filter ADX/ATR)")

    keep_cols = ["datetime", "open", "high", "low", "close", "atr", "adx", "bull_chain", "bear_chain",
                 "v12_raw_score_nofilter"]
    df_scored = df_scored[keep_cols]
    df_scored.to_parquet(SCORED_ALL_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {SCORED_ALL_CACHE_PATH}")

print(f"\nTotal candle: {len(df_scored)}")

Load skor dari cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v19\df_2019_2024_scored_nofilter.parquet

Total candle: 411067


## 2. Backtest engine — semua parameter kandidat bisa divariasikan

In [ ]:
def check_h1_alignment(row, direction):
    h1_ema_50 = row.get("h1_ema_50")
    h1_ema_200 = row.get("h1_ema_200")
    if h1_ema_50 is None or h1_ema_200 is None or pd.isna(h1_ema_50) or pd.isna(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def calc_lot(equity: float, risk_pct: float, sl_distance: float) -> float:
    if sl_distance <= 0 or equity <= 0:
        return MIN_LOT
    risk_amount = equity * risk_pct
    raw_lot = risk_amount / (sl_distance * CONTRACT_SIZE)
    lot = math.floor(raw_lot / LOT_STEP) * LOT_STEP
    return max(round(lot, 2), MIN_LOT)


def run_backtest_regime(
    df_signals: pd.DataFrame,
    adx_min: float,
    atr_min_pct: float,
    min_signal_score: float,
    sl_mult: float,
    tp_mult: float,
    max_hold: int | float,
    min_atr_over_spread: float,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    """Backtest full-parametrized -- SEMUA nilai yang biasanya konstanta v13 di sini jadi
    argumen, supaya bisa grid search bebas utk cari kombinasi yang cocok rezim volatilitas
    rendah. Kolom H1 (h1_ema_50/200) TIDAK ada di cache v19 (cuma disimpan kolom minimal) --
    check_h1_alignment akan selalu True (skip filter) krn row.get() balikin None -- INI
    KETERBATASAN yg dicatat eksplisit di kesimpulan, bukan disembunyikan."""
    max_hold = int(max_hold)
    close_arr = df_signals["close"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_raw_score_nofilter"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    close_full_arr = df_signals["close"].to_numpy()

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    n = len(df_signals)

    while i < n:
        adx = adx_arr[i]
        atr = atr_arr[i]
        close = close_arr[i]
        score = score_arr[i]

        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue
        if close > 0 and (atr / close * 100) < atr_min_pct:
            i += 1
            continue
        if min_atr_over_spread > 0 and (atr / spread_points) < min_atr_over_spread:
            i += 1
            continue

        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        else:
            i += 1
            continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        entry_time = datetime_arr[i]

        if direction == "BUY":
            tp_price, sl_price = entry_price + tp_points, entry_price - sl_points
        else:
            tp_price, sl_price = entry_price - tp_points, entry_price + sl_points

        lot = calc_lot(equity, RISK_PCT, sl_points)
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_full_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

## 3. Split TRAIN (2019-2023) / TEST (2024) & baseline v13 parameter (sbg pembanding)

**Catatan keterbatasan penting**: cache v19 (dibuat khusus riset ini) TIDAK menyimpan kolom H1
(`h1_ema_50`/`h1_ema_200`) -- jadi filter H1 trend alignment yang biasanya jadi bagian
`generate_signal_v12` TIDAK diterapkan di sini. Ini beda dari v13 asli. Dicatat eksplisit,
bukan disembunyikan -- kalau kombinasi yang ditemukan di v19 terlihat menjanjikan, langkah
selanjutnya WAJIB validasi ulang DENGAN filter H1 sebelum dipertimbangkan lebih lanjut.

In [4]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")

df_train = df_scored[df_scored["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df_scored[df_scored["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle, {df_train['datetime'].min()} -> {df_train['datetime'].max()}")
print(f"TEST (2024): {len(df_test)} candle, {df_test['datetime'].min()} -> {df_test['datetime'].max()}")

# Baseline: parameter v13 ASLI diterapkan apa adanya ke rezim rendah (dgn spread realistis) --
# ini pembanding utk tahu seberapa BURUK parameter v13 kalau dipaksakan, sebelum grid search.
baseline_params = dict(
    adx_min=18.0, atr_min_pct=0.03, min_signal_score=9.0,
    sl_mult=2.0, tp_mult=4.0, max_hold=12, min_atr_over_spread=0.0,
)
trades_baseline_train = run_backtest_regime(df_train, **baseline_params)
trades_baseline_test = run_backtest_regime(df_test, **baseline_params)
print("\n=== Baseline (parameter v13 asli, dipaksakan ke rezim rendah, TANPA filter H1) ===")
print("TRAIN:", evaluate(trades_baseline_train, INITIAL_EQUITY))
print("TEST:", evaluate(trades_baseline_test, INITIAL_EQUITY))

TRAIN (2019-2023): 351136 candle, 2019-01-01 23:00:00+00:00 -> 2023-12-29 21:55:00+00:00
TEST (2024): 59931 candle, 2024-01-01 23:00:00+00:00 -> 2024-12-30 23:55:00+00:00



=== Baseline (parameter v13 asli, dipaksakan ke rezim rendah, TANPA filter H1) ===
TRAIN: {'total_trades': 7087, 'win_rate_pct': 11.58, 'profit_factor': np.float64(0.24), 'net_pnl': np.float64(-10570.34), 'max_drawdown_pct': np.float64(-10570.34)}
TEST: {'total_trades': 1203, 'win_rate_pct': 22.36, 'profit_factor': np.float64(0.42), 'net_pnl': np.float64(-1656.1), 'max_drawdown_pct': np.float64(-1656.1)}


## 4. Grid search di TRAIN — cari kombinasi parameter yang cocok rezim rendah

Ruang parameter dibatasi (bukan full cartesian product besar, backtest sequential per baris di
351K candle TRAIN per kombinasi -- grid besar akan makan waktu lama). Diarahkan scr teori:
`min_signal_score` lebih rendah dari 9.0 (skor absolut cenderung lebih kecil di ATR kecil),
`sl_mult`/`tp_mult` lebih besar (supaya SL/TP dlm poin absolut cukup lebar dibanding spread 1.82
meski ATR kecil), `min_atr_over_spread` lebih longgar dari 3.0 (v18 dicari dari rezim TINGGI).

In [5]:
import itertools
import time as _time

GRID = {
    "adx_min": [15.0, 18.0],
    "atr_min_pct": [0.0],
    "min_signal_score": [6.0, 7.5, 9.0],
    "sl_mult": [2.0, 3.0],
    "tp_mult": [4.0, 6.0],
    "max_hold": [12],
    "min_atr_over_spread": [0.0, 1.5, 2.0],
}

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_regime(df_train, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s, n={metrics['total_trades']}, PF={metrics['profit_factor']}")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 10 kandidat dgn sample TRAIN >= {MIN_SAMPLE_TRAIN} trade ===")
print(grid_valid.head(10)[["total_trades", "win_rate_pct", "profit_factor", "max_drawdown_pct",
                            "adx_min", "atr_min_pct", "min_signal_score", "sl_mult", "tp_mult",
                            "max_hold", "min_atr_over_spread"]])

Total kombinasi grid: 72


  [1/72] 1s, n=34253, PF=0.15


  [2/72] 1s, n=1251, PF=0.61


  [3/72] 2s, n=409, PF=0.75


  [4/72] 3s, n=34045, PF=0.15


...

  [68/72] 48s, n=441, PF=0.85


  [69/72] 48s, n=137, PF=0.86


  [70/72] 49s, n=6785, PF=0.31


  [71/72] 50s, n=436, PF=0.91


  [72/72] 50s, n=135, PF=0.9

Grid search selesai dalam 50s

=== Top 10 kandidat dgn sample TRAIN >= 30 trade ===
    total_trades  win_rate_pct  profit_factor  max_drawdown_pct  adx_min  atr_min_pct  min_signal_score  sl_mult  tp_mult  max_hold  min_atr_over_spread
70           436         44.95           0.91           -268.76     18.0          0.0               9.0      3.0      6.0        12                  1.5
34           443         44.92           0.90           -283.79     15.0          0.0               9.0      3.0      6.0        12                  1.5
35           137         43.07           0.90           -107.71     15.0          0.0               9.0      3.0      6.0        12                  2.0
71           135         42.96           0.90           -104.30     18.0          0.0               9.0      3.0      6.0        12                  2.0
56           262         43.13           0.87           -134.26     18.0          0.0               7.5      3.0      4.0

## 5. Kesimpulan

**Jawaban jujur: TIDAK DITEMUKAN kombinasi parameter yang profitable untuk rezim volatilitas
rendah (2019-2024), dengan spread broker realistis (1.82).**

**Bukti**: dari 72 kombinasi grid search (rentang `adx_min` 15-18, `min_signal_score` 6.0-9.0,
`sl_mult`/`tp_mult` 2-3x/4-6x, `min_atr_over_spread` 0-2.0) di data TRAIN (2019-2023, 351K
candle) — **TIDAK SATU PUN mencapai profit factor > 1.0**. Kandidat terbaik (`adx_min=18,
min_signal_score=9.0, sl_mult=3.0, tp_mult=6.0, min_atr_over_spread=1.5`) cuma PF=0.91 (masih
rugi bersih ~9% dari total volume trading, dengan 436 trade — sample besar & bisa dipercaya,
bukan kebetulan sample kecil). Baseline (parameter v13 asli dipaksakan) jauh lebih buruk lagi
(PF=0.24 TRAIN, 0.42 TEST) — jadi meski grid search memperbaiki cukup jauh dari baseline, tetap
tidak sampai ke titik profitable.

**Kenapa TIDAK dilanjutkan ke validasi TEST out-of-sample**: kriteria keberhasilan yang
ditetapkan di awal (PF>1.5 di TEST) sudah gagal terpenuhi bahkan di TRAIN (PF maksimal cuma
0.91) — melanjutkan ke TEST tanpa kandidat yang layak akan sia-sia (fishing for significance:
mencari-cari sampai ketemu satu yang kebetulan bagus di TEST, padahal TRAIN-nya sendiri tidak
solid). Sesuai kriteria yang ditetapkan sblm riset dimulai, kesimpulan JUJURnya adalah rezim ini
memang tidak layak ditradingkan dgn edge yang bisa ditemukan dari mesin scoring v12 yang ada.

**Kenapa ini masuk akal, bukan kegagalan riset**: v18 sudah membuktikan akar masalahnya scr
matematis — 58.5% sinyal historis terjadi saat ATR lebih kecil dari spread broker (1.82). Di
rezim 2019-2024, ATR rata-rata cuma $0.85-1.85 (v17) — bahkan dgn `sl_mult`/`tp_mult` dinaikkan
jadi 3x/6x (mencoba memperlebar SL/TP scr proporsional), margin thd spread tetap terlalu tipis
utk mengalahkan biaya transaksi secara konsisten. Ini BUKAN soal salah pilih parameter, tapi
soal STRUKTUR EKONOMI trading M5 scalping di rezim volatilitas serendah itu, dgn spread broker
lokal (MIFX) yang relatif lebar dibanding ATR M5.

**Rekomendasi (bukan keputusan otomatis):**
1. **Robot SEHARUSNYA memang tidak trading sama sekali** kalau market kembali ke rezim
   volatilitas serendah 2019-2024 -- bukan krn bug atau kekurangan riset, tapi krn TIDAK ADA
   edge yang valid ditemukan setelah pencarian yang cukup menyeluruh (72 kombinasi, kriteria
   sample minimum diterapkan, dibandingkan baseline). Filter `min_atr_over_spread` (v18) yang
   secara otomatis men-skip entry di kondisi ini justru berperilaku BENAR, bukan terlalu ketat.
2. **Kalau rezim rendah kembali terjadi di masa depan**, robot idealnya diam total (equity flat,
   nunggu volatilitas naik lagi) drpd dipaksakan trading dgn edge yang tidak terbukti ada.
   Ini konsisten dgn prinsip "mending gak trading drpd trading rugi" yang mendasari riset v19 ini.
3. **TIDAK ADA strategi baru yang ditambahkan ke `usecase.py`/`params.json`** dari v19 -- hasil
   negatif (tidak ditemukan) itu SENDIRI adalah hasil yang valid & berharga, mencegah kita
   memaksakan strategi yang keliatan "bekerja" di TRAIN tapi sebenarnya cuma kebetulan/overfitting
   (pelajaran yang sudah dipetik berkali-kali di riset2 sebelumnya, terutama v15/DSR).
4. Kalau di masa depan ada motivasi kuat utk tetap trading di rezim rendah, jalur yang LEBIH
   MASUK AKAL bukan re-tuning parameter existing (sudah dicoba di v19, gagal), tapi eksplorasi
   LOGIKA SINYAL BERBEDA scr fundamental (mis. mean-reversion/range-trading, sesuai tabel yang
   pernah didiskusikan) -- itu riset terpisah & lebih besar, di luar cakupan v19.

**Keterbatasan**: (a) grid search TIDAK exhaustive (72 kombinasi terarah scr teori, bukan full
cartesian ratusan/ribuan) -- kemungkinan kecil ada kombinasi di luar ruang yang dicoba yang
lolos, tapi mengingat baseline & smua kandidat konsisten di bawah PF 1.0, kemungkinan itu kecil;
(b) filter H1 trend alignment TIDAK diterapkan (keterbatasan data cache v19, lihat catatan
Section 3) -- H1 alignment biasanya MENGURANGI jumlah trade & bisa jadi menaikkan kualitas,
tapi mengingat gap dari PF 0.91 ke 1.5 (kriteria sukses) cukup jauh, kecil kemungkinan H1
alignment sendirian bisa menutup gap itu; (c) exhaustion-mode (momentum chain 8/8) TIDAK
diikutsertakan -- fitur v13 yang spesifik utk rezim tinggi, belum tentu relevan tapi belum diuji
scr eksplisit di rezim rendah.